# Part 3 · Notebook 02 — Functions, control flow and files

**Sessions:** S2 (Control flow, functions & comprehensions) · S4 (Files, exceptions & modules) · Clinic W1 · [Lesson plan](../../docs/lessons/PART_03_PYTHON_ENGINEERING.md) · graded labs in [`labs/part03/`](../../labs/part03/)

**You will:**
1. Write small, pure functions for returns and drawdown, and compare them with NumPy.
2. Route broker messages with `match`.
3. Avoid the mutable-default trap.
4. Parse a messy trade log, collecting an error per bad line instead of crashing.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p3lib.py is in notebooks/part03/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p3lib as p

p.use_course_style()

## 1. Returns and drawdown on plain lists

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
prices = [100.0, 102.0, 99.0, 104.0, 101.0, 107.0]
# ✍️ simple returns with a list comprehension over consecutive pairs (hint: zip(prices, prices[1:]))
rets = ...
rets = p.check("simple returns", rets, p.simple_returns(prices))
rets

In [ ]:
def max_drawdown(prices):
    peak, mdd = prices[0], 0.0
    for px in prices:
        # ✍️ update the running peak and the worst drawdown so far (px / peak − 1)
        ...
    return mdd

mdd = p.check("max drawdown", max_drawdown(prices), p.max_drawdown(prices))
mdd

In [ ]:
big = list(p.prices_array(200_000))
arr = np.asarray(big)
t_py = p.timeit(p.max_drawdown, big, repeat=3)
t_np = p.timeit(lambda x: p.drawdown_vectorized(x).min(), arr, repeat=3)
print(f"pure Python {t_py * 1e3:.1f} ms   NumPy {t_np * 1e3:.2f} ms   ({t_py / t_np:.0f}× faster)")
print("Same answer:", np.isclose(p.max_drawdown(big), p.drawdown_vectorized(arr).min()))

## 2. Routing broker messages with `match`

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def route(msg: dict) -> str:
    match msg:
        case {"type": "fill", "qty": q} if q > 0:
            return "book_fill"
        case {"type": "fill"}:
            return "reject_bad_fill"
        # ✍️ add: "cancel" or "reject" → "release_order"; "heartbeat" → "ignore"; anything else → "log_unknown"
        case _:
            return ...

msgs = [{"type": "fill", "qty": 100}, {"type": "fill", "qty": 0}, {"type": "cancel", "id": 7},
        {"type": "reject", "id": 8}, {"type": "heartbeat"}, {"type": "news"}]
routed = [route(m) for m in msgs]
routed = p.check("message routing", routed, [p.route(m) for m in msgs])
routed

## 3. The mutable-default trap

In [ ]:
def add_fill(fill, book=[]):               # ❌ the list is created ONCE, when the function is defined
    book.append(fill)
    return book

print(add_fill("A"), add_fill("B"))           # the second call still sees "A"

def add_fill_ok(fill, book=None):             # ✅
    book = [] if book is None else book
    book.append(fill)
    return book

print(add_fill_ok("A"), add_fill_ok("B"))

## 4. Parsing a messy trade log (clinic W1)

Fail loudly per line, but keep going: collect `(line number, message)` for every bad line.

In [ ]:
print(p.TRADE_CSV)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import csv, io
from decimal import InvalidOperation

def parse_trades(text):
    rows, errors = [], []
    for i, r in enumerate(csv.DictReader(io.StringIO(text)), start=2):    # line 1 is the header
        try:
            # ✍️ raise ValueError(f"bad side {r['side']!r}") unless side is BUY or SELL
            ...
            qty = int(r["qty"])
            if not r["price"]:
                raise ValueError("missing price")
            price = Decimal(r["price"])
            rows.append({"time": datetime.fromisoformat(r["time"]), "symbol": r["symbol"], "side": r["side"],
                         "qty": qty, "price": price})
        except (ValueError, InvalidOperation) as e:
            # ✍️ record (i, str(e)) in errors
            ...
    return rows, errors

from datetime import datetime
parsed = parse_trades(p.TRADE_CSV)
rows, errors = p.check("trade-log parser", parsed, p.parse_trades(p.TRADE_CSV))
errors

In [ ]:
import json
report = {"valid_rows": len(rows), "errors": errors,
          "realized_pnl": {s: str(v) for s, v in p.realized_pnl(rows).items()}}      # Decimal → str keeps it exact
print(json.dumps(report, indent=2))

## Questions
1. Why is a bare `except:` dangerous in trading code? What should happen to an unknown message type?
2. Which function in this notebook has a side effect, and how would you test it?
3. The report stores P&L as strings. Why not floats?

**Graded version:** `labs/part03/week07_basics` and the clinic CLI in `labs/part03/clinic_w1_trade_log`.